In [10]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve().parent 
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

results_dir = project_root / 'results'
results_dir.mkdir(exist_ok=True)

# Experimentation with PyOD Models

This notebook evaluates PyOD models on two UCI datasets that are more suitable for anomaly detection: `shuttle` and `arrhythmia`. The goal is to load the datasets from the RADAR static dataset module, reframe them as anomaly-detection benchmarks, and compare several PyOD models using label-based and score-based metrics.

In [2]:
# List all available PyOD algorithms in RADAR
from RADAR.static_data.algorithms.pyod import pyod_algorithms

print("Available PyOD algorithms in RADAR:")
print("=" * 50)
for i, name in enumerate(sorted(pyod_algorithms.keys()), 1):
    print(f"{i:2}. {name}")

print(f"\nTotal available models: {len(pyod_algorithms)}")

2026-03-22 19:42:57.739451: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-22 19:42:58.013918: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774204978.220451   15017 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774204978.289493   15017 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-22 19:42:58.546977: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

Available PyOD algorithms in RADAR:
 1. abod
 2. ae1svm
 3. alad
 4. anogan
 5. auto_encoder
 6. cblof
 7. cd
 8. cof
 9. copod
10. deep_svdd
11. devnet
12. dif
13. ecod
14. feature_bagging
15. gmm
16. hbos
17. iforest
18. inne
19. kde
20. knn
21. kpca
22. lmdd
23. loci
24. loda
25. lof
26. lscp
27. lunar
28. mad
29. mcd
30. mo_gaal
31. ocsvm
32. pca
33. qmcd
34. rgraph
35. rod
36. sampling
37. so_gaal
38. sod
39. sos
40. suod
41. vae
42. xgbod

Total available models: 42


In [4]:
# Import Required Libraries
import importlib
import numpy as np
import pandas as pd
from RADAR.static_data.algorithms import pyod
import RADAR.metrics_module as metrics_module
metrics_module = importlib.reload(metrics_module)

In [5]:
import RADAR.static_data.anomaly_dataset_utils as anomaly_dataset_utils

anomaly_dataset_utils = importlib.reload(anomaly_dataset_utils)

uci_dataset_configs = {
    "shuttle": anomaly_dataset_utils.build_loaded_uci_anomaly_dataset(
        dataset_name="shuttle",
        normal_label=1,
        target_test_contamination=0.1,
        max_train_normals=8000,
        max_test_size=5000,
    ),
    "arrhythmia": anomaly_dataset_utils.build_loaded_uci_anomaly_dataset(
        dataset_name="arrhythmia",
        normal_label=1,
        target_test_contamination=0.1,
    ),
}

In [ ]:
# Load and print the original dataset details
from RADAR.static_data.anomaly_dataset_utils import load_dataset_silently

# Load the raw dataset for arrhythmia
def print_raw_dataset(dataset_name):
    X_raw, y_raw = load_dataset_silently(dataset_name)
    print(f"Original Dataset: {dataset_name}")
    print(f"X_raw shape: {X_raw.shape}")
    print(f"y_raw distribution: {np.bincount(y_raw)}")
    print(y_raw)  
# Call the function for arrhythmia
print_raw_dataset("arrhythmia")


In [6]:
uci_summary_rows = []
for dataset_name, config in uci_dataset_configs.items():
    uci_summary_rows.append(
        {
            "dataset": dataset_name,
            "samples": config["n_samples"],
            "features": config["n_features"],
            "original_anomaly_ratio": round(config["original_positive_ratio"], 4),
            "benchmark_test_contamination": round(
                config["benchmark_test_positive_ratio"], 4
            ),
            "train_normals_used": config["train_normals"],
            "test_normals": config["test_normals"],
            "test_anomalies": config["test_anomalies"],
        }
    )

uci_summary_df = pd.DataFrame(uci_summary_rows)
display(uci_summary_df)

,dataset,samples,features,original_anomaly_ratio,benchmark_test_contamination,train_normals_used,test_normals,test_anomalies
0,shuttle,58000,7,0.214,0.1036,8000,4482,518
1,arrhythmia,452,279,0.458,0.0926,196,49,5


### PyOD Experiment on Shuttle and Arrhythmia



In [7]:
from pyod.models.abod import ABOD
from pyod.models.alad import ALAD
from pyod.models.anogan import AnoGAN
from pyod.models.cblof import CBLOF
from pyod.models.feature_bagging import FeatureBagging
from pyod.models.gmm import GMM
from pyod.models.hbos import HBOS
from pyod.models.iforest import IForest
from pyod.models.inne import INNE
from pyod.models.kde import KDE
from pyod.models.knn import KNN
from pyod.models.lmdd import LMDD
from pyod.models.lof import LOF
from pyod.models.lscp import LSCP
from pyod.models.mcd import MCD
from pyod.models.ocsvm import OCSVM
from pyod.models.pca import PCA
import time

uci_pyod_models = [
    {"algorithm_": "abod"},
    {"algorithm_": "alad", "epochs": 30, "verbose": 0},
    {"algorithm_": "anogan", "epochs": 30, "verbose": 0},
    {"algorithm_": "cblof"},
    {"algorithm_": "feature_bagging"},
    {"algorithm_": "gmm"},
    {"algorithm_": "hbos"},
    {"algorithm_": "iforest", "random_state": 42},
    {"algorithm_": "inne", "random_state": 42},
    {"algorithm_": "kde"},
    {"algorithm_": "knn", "n_neighbors": 5},
    {"algorithm_": "lmdd"},
    {"algorithm_": "lof", "n_neighbors": 5},
    {"algorithm_": "lscp", "detector_list": [LOF(), LOF()]},
    {"algorithm_": "mcd"},
    {"algorithm_": "ocsvm"},
    {"algorithm_": "pca"},
]

# Direct PyOD classes for comparison
direct_pyod_original = {
   "abod": ABOD,
    "alad": ALAD,
    "anogan": AnoGAN,
    "cblof": CBLOF,
    "feature_bagging": FeatureBagging,
    "gmm": GMM,
    "hbos": HBOS,
    "iforest": IForest,
    "inne": INNE,
    "kde": KDE,
    "knn": KNN,
    "lmdd": LMDD,
    "lof": LOF,
    "lscp": LSCP,
    "mcd": MCD,
    "ocsvm": OCSVM,
    "pca": PCA,
}

uci_results = []

for dataset_name, config in uci_dataset_configs.items():
    print(f"\nDataset: {dataset_name}")
    print(
        f"Training with normal-only samples: {config['train_normals']} | "
        f"Benchmark contamination: {config['benchmark_test_positive_ratio']:.3f}"
    )

    for model_params in uci_pyod_models:
        algorithm_name = model_params["algorithm_"]
        shared_kwargs = {k: v for k, v in model_params.items() if k != "algorithm_"}
        
        model_kwargs = {
            **model_params,
            "contamination": config["benchmark_test_positive_ratio"],
        }

        # === RADAR Platform timing ===
        model = pyod.PyodAnomalyDetection(**model_kwargs)
        start_platform = time.time()
        model.fit(config["X_train"])
        platform_fit_time = time.time() - start_platform
        
        start_predict = time.time()
        predictions = np.asarray(model.predict(config["X_test"])).astype(int).ravel()
        scores = np.asarray(model.decision_function(config["X_test"])).ravel()
        platform_predict_time = time.time() - start_predict
        platform_total_time = platform_fit_time + platform_predict_time

        # === Direct PyOD timing ===
        direct_cls = direct_pyod_original[algorithm_name]
        direct_model = direct_cls(
            contamination=config["benchmark_test_positive_ratio"],
            **shared_kwargs
        )
        start_direct = time.time()
        direct_model.fit(config["X_train"])
        direct_fit_time = time.time() - start_direct
        
        start_direct_pred = time.time()
        _ = direct_model.predict(config["X_test"])
        _ = direct_model.decision_function(config["X_test"])
        direct_predict_time = time.time() - start_direct_pred
        direct_total_time = direct_fit_time + direct_predict_time

        # Calculate overhead and speedup
        overhead = platform_total_time - direct_total_time
        speedup = direct_total_time / platform_total_time if platform_total_time > 0 else np.nan

        # Metrics
        accuracy = metrics_module.metric_accuracy(config["y_test"], predictions) / 100
        precision = metrics_module.metric_precision(config["y_test"], predictions)
        recall = metrics_module.metric_recall(config["y_test"], predictions)
        f1 = metrics_module.metric_F1score(config["y_test"], predictions)

        finite_scores = np.isfinite(scores)
        if finite_scores.all():
            roc_auc = metrics_module.metric_AUC_ROC_scores(config["y_test"], scores)
            pr_auc = metrics_module.metric_PR_AUC(config["y_test"], scores)
            score_note = ""
        else:
            roc_auc = np.nan
            pr_auc = np.nan
            score_note = " | score metrics skipped (NaN decision scores)"

        print(f"\nModel: {algorithm_name}{score_note}")
        metrics_module.print_metrics(["Accuracy", "Precision", "Recall", "F1"], config["y_test"], predictions)
        if finite_scores.all():
            print(f"ROC AUC: {roc_auc:.3f} | PR AUC: {pr_auc:.3f}")
        print(f"Platform: {platform_total_time:.4f}s | Direct: {direct_total_time:.4f}s | Overhead: {overhead:.4f}s | Speedup: {speedup:.4f}")

        uci_results.append({
            "dataset": dataset_name,
            "algorithm": algorithm_name,
            "category": "original",
            "contamination": round(config["benchmark_test_positive_ratio"], 4),
            "accuracy": round(accuracy, 4),
            "precision": round(precision, 4),
            "recall": round(recall, 4),
            "f1": round(f1, 4),
            "roc_auc": round(float(roc_auc), 4) if np.isfinite(roc_auc) else np.nan,
            "pr_auc": round(float(pr_auc), 4) if np.isfinite(pr_auc) else np.nan,
            "platform_time_s": round(platform_total_time, 4),
            "direct_time_s": round(direct_total_time, 4),
            "overhead_s": round(overhead, 4),
            "speedup": round(speedup, 4) if not np.isnan(speedup) else np.nan,
        })

uci_results_df = pd.DataFrame(uci_results).sort_values(
    ["dataset", "pr_auc", "roc_auc"],
    ascending=[True, False, False],
    na_position="last",
).reset_index(drop=True)

display(uci_results_df)


Dataset: shuttle
Training with normal-only samples: 8000 | Benchmark contamination: 0.104

Model: abod | score metrics skipped (NaN decision scores)
Accuracy: 89.640%
Precision: 0.000
Recall: 0.000
F1 Score: 0.000
Platform: 3.6897s | Direct: 2.0658s | Overhead: 1.6239s | Speedup: 0.5599

Model: alad
Accuracy: 81.800%
Precision: 0.160
Recall: 0.178
F1 Score: 0.168
ROC AUC: 0.578 | PR AUC: 0.156
Platform: 1.0066s | Direct: 0.4188s | Overhead: 0.5878s | Speedup: 0.4160

Model: anogan
Accuracy: 46.500%
Precision: 0.154
Recall: 0.929
F1 Score: 0.265
ROC AUC: 0.785 | PR AUC: 0.554
Platform: 353.4991s | Direct: 346.0588s | Overhead: 7.4403s | Speedup: 0.9790

Model: cblof
Accuracy: 90.920%
Precision: 0.534
Recall: 0.963
F1 Score: 0.687
ROC AUC: 0.974 | PR AUC: 0.807
Platform: 2.6374s | Direct: 1.7173s | Overhead: 0.9201s | Speedup: 0.6511

Model: feature_bagging
Accuracy: 80.360%
Precision: 0.052
Recall: 0.052
F1 Score: 0.052
ROC AUC: 0.719 | PR AUC: 0.185
Platform: 3.9613s | Direct: 4.3763s

,dataset,algorithm,category,contamination,accuracy,precision,recall,f1,roc_auc,pr_auc,platform_time_s,direct_time_s,overhead_s,speedup
0,arrhythmia,hbos,original,0.0926,0.7963,0.2500,0.6000,0.3529,0.8286,0.5417,0.0393,0.0378,0.0015,0.9630
1,arrhythmia,pca,original,0.0926,0.7593,0.2143,0.6000,0.3158,0.7673,0.5309,0.0397,0.0359,0.0038,0.9031
2,arrhythmia,alad,original,0.0926,0.8704,0.2500,0.2000,0.2222,0.8000,0.5194,2.0851,1.5017,0.5835,0.7202
3,arrhythmia,knn,original,0.0926,0.7593,0.2143,0.6000,0.3158,0.7755,0.4641,0.0459,0.0155,0.0304,0.3384
4,arrhythmia,cblof,original,0.0926,0.7407,0.2000,0.6000,0.3000,0.7714,0.4629,0.4060,0.3679,0.0380,0.9064
5,arrhythmia,ocsvm,original,0.0926,0.7222,0.1875,0.6000,0.2857,0.7510,0.4626,0.0089,0.0081,0.0008,0.9058
6,arrhythmia,abod,original,0.0926,0.8333,0.2500,0.4000,0.3077,0.7510,0.4590,0.0914,0.0594,0.0320,0.6496
7,arrhythmia,lscp,original,0.0926,0.7778,0.2308,0.6000,0.3333,0.7714,0.4263,0.8603,0.8918,-0.0315,1.0366
8,arrhythmia,anogan,original,0.0926,0.0926,0.0769,0.8000,0.1404,0.7265,0.4252,34.5219,28.8233,5.6986,0.8349
9,arrhythmia,feature_bagging,original,0.0926,0.7963,0.2500,0.6000,0.3529,0.7633,0.4227,0.3459,0.2544,0.0915,0.7355


### Experiment: Other classic models

In [ ]:
# New classical models 
from pyod.models.copod import COPOD
from pyod.models.ecod import ECOD
from pyod.models.cof import COF
from pyod.models.sod import SOD
from pyod.models.sos import SOS
from pyod.models.loda import LODA
from pyod.models.loci import LOCI
from pyod.models.kpca import KPCA
from pyod.models.rod import ROD
from pyod.models.qmcd import QMCD
from pyod.models.sampling import Sampling
from pyod.models.cd import CD
from pyod.models.rgraph import RGraph
from pyod.models.suod import SUOD

new_classical_models = [
    {"algorithm_": "copod"},           # Copula-based, very fast, parameter-free
    {"algorithm_": "ecod"},            # Empirical CDF-based, efficient
    {"algorithm_": "cof", "n_neighbors": 5},  # Connectivity-based
    {"algorithm_": "sod", "n_neighbors": 5},  # Subspace Outlier Detection
    {"algorithm_": "sos"},             # Stochastic Outlier Selection
    {"algorithm_": "loda"},            # Lightweight Online Detector
    {"algorithm_": "loci"},            # Local Correlation Integral
    {"algorithm_": "kpca"},            # Kernel PCA
    {"algorithm_": "rod"},             # Rotation-based
    {"algorithm_": "qmcd"},            # Quasi-Monte Carlo Discrepancy
    {"algorithm_": "sampling"},        # Sampling-based
    {"algorithm_": "cd"},              # Cook's Distance
    {"algorithm_": "rgraph"},          # R-graph
    {"algorithm_": "suod"},            # Scalable Unsupervised OD
]

# Direct PyOD classes for comparison
direct_pyod_classical = {
    "copod": COPOD,
    "ecod": ECOD,
    "cof": COF,
    "sod": SOD,
    "sos": SOS,
    "loda": LODA,
    "loci": LOCI,
    "kpca": KPCA,
    "rod": ROD,
    "qmcd": QMCD,
    "sampling": Sampling,
    "cd": CD,
    "rgraph": RGraph,
    "suod": SUOD,
}

new_classical_results = []

for dataset_name, config in uci_dataset_configs.items():
    print(f"\n{'='*60}")
    print(f"Dataset: {dataset_name}")
    print(f"Training samples: {config['train_normals']} | Test contamination: {config['benchmark_test_positive_ratio']:.3f}")
    print("="*60)

    for model_params in new_classical_models:
        algorithm_name = model_params["algorithm_"]
        shared_kwargs = {k: v for k, v in model_params.items() if k != "algorithm_"}
        
        try:
            model_kwargs = {
                **model_params,
                "contamination": config["benchmark_test_positive_ratio"],
            }

            # === RADAR Platform timing ===
            model = pyod.PyodAnomalyDetection(**model_kwargs)
            start_platform = time.time()
            model.fit(config["X_train"])
            platform_fit_time = time.time() - start_platform
            
            start_predict = time.time()
            predictions = np.asarray(model.predict(config["X_test"])).astype(int).ravel()
            scores = np.asarray(model.decision_function(config["X_test"])).ravel()
            platform_predict_time = time.time() - start_predict
            platform_total_time = platform_fit_time + platform_predict_time

            # === Direct PyOD timing ===
            direct_cls = direct_pyod_classical[algorithm_name]
            direct_model = direct_cls(
                contamination=config["benchmark_test_positive_ratio"],
                **shared_kwargs
            )
            start_direct = time.time()
            direct_model.fit(config["X_train"])
            direct_fit_time = time.time() - start_direct
            
            start_direct_pred = time.time()
            _ = direct_model.predict(config["X_test"])
            _ = direct_model.decision_function(config["X_test"])
            direct_predict_time = time.time() - start_direct_pred
            direct_total_time = direct_fit_time + direct_predict_time

            # Calculate overhead and speedup
            overhead = platform_total_time - direct_total_time
            speedup = direct_total_time / platform_total_time if platform_total_time > 0 else np.nan

            # Metrics
            accuracy = metrics_module.metric_accuracy(config["y_test"], predictions) / 100
            precision = metrics_module.metric_precision(config["y_test"], predictions)
            recall = metrics_module.metric_recall(config["y_test"], predictions)
            f1 = metrics_module.metric_F1score(config["y_test"], predictions)

            finite_scores = np.isfinite(scores)
            if finite_scores.all():
                roc_auc = metrics_module.metric_AUC_ROC_scores(config["y_test"], scores)
                pr_auc = metrics_module.metric_PR_AUC(config["y_test"], scores)
            else:
                roc_auc = np.nan
                pr_auc = np.nan

            print(f"\n{algorithm_name}: F1={f1:.3f}, ROC-AUC={roc_auc:.3f} | Platform: {platform_total_time:.4f}s, Direct: {direct_total_time:.4f}s, Overhead: {overhead:.4f}s")

            new_classical_results.append({
                "dataset": dataset_name,
                "algorithm": algorithm_name,
                "category": "classical_new",
                "contamination": round(config["benchmark_test_positive_ratio"], 4),
                "accuracy": round(accuracy, 4),
                "precision": round(precision, 4),
                "recall": round(recall, 4),
                "f1": round(f1, 4),
                "roc_auc": round(float(roc_auc), 4) if np.isfinite(roc_auc) else np.nan,
                "pr_auc": round(float(pr_auc), 4) if np.isfinite(pr_auc) else np.nan,
                "platform_time_s": round(platform_total_time, 4),
                "direct_time_s": round(direct_total_time, 4),
                "overhead_s": round(overhead, 4),
                "speedup": round(speedup, 4) if not np.isnan(speedup) else np.nan,
            })

        except Exception as e:
            print(f"\n{algorithm_name}: Error - {str(e)[:80]}")
            new_classical_results.append({
                "dataset": dataset_name,
                "algorithm": algorithm_name,
                "category": "classical_new",
                "contamination": round(config["benchmark_test_positive_ratio"], 4),
                "accuracy": np.nan,
                "precision": np.nan,
                "recall": np.nan,
                "f1": np.nan,
                "roc_auc": np.nan,
                "pr_auc": np.nan,
                "platform_time_s": np.nan,
                "direct_time_s": np.nan,
                "overhead_s": np.nan,
                "speedup": np.nan,
                "error": str(e)[:100],
            })

new_classical_df = pd.DataFrame(new_classical_results).sort_values(
    ["dataset", "f1"], ascending=[True, False]
).reset_index(drop=True)

print("\n" + "="*60)
print("Results Summary - New Classical Models")
print("="*60)
display(new_classical_df)

### Experiment: Deep Learning Models (Slower)

**Warning**: These models require more computational resources and may take several minutes to train. They use neural networks (TensorFlow/Keras backend).

In [ ]:
# Deep Learning models - require more computation time
# Note: These models use TensorFlow/Keras backend
from pyod.models.auto_encoder import AutoEncoder
from pyod.models.vae import VAE
from pyod.models.deep_svdd import DeepSVDD
from pyod.models.ae1svm import AE1SVM
from pyod.models.dif import DIF
from pyod.models.lunar import LUNAR

DEEP_LEARNING_MODELS = [
    {"algorithm_": "auto_encoder", "epoch_num": 30, "verbose": 0},  # uses epoch_num
    {"algorithm_": "vae", "epoch_num": 30, "verbose": 0},            # uses epoch_num
    {"algorithm_": "deep_svdd", "epochs": 30, "verbose": 0},         # n_features injected at runtime
    {"algorithm_": "ae1svm", "epochs": 30},                           # no verbose support
    {"algorithm_": "dif"},
    {"algorithm_": "lunar"},
]
# Direct PyOD classes for comparison
direct_pyod_deep = {
    "auto_encoder": AutoEncoder,
    "vae": VAE,
    "deep_svdd": DeepSVDD,
    "ae1svm": AE1SVM,
    "dif": DIF,
    "lunar": LUNAR,
}

# GAN-based models (can be unstable, optional)
gan_models = [
    {"algorithm_": "so_gaal", "epochs": 20, "verbose": 0},
    {"algorithm_": "mo_gaal", "epochs": 20, "verbose": 0},
]

# DevNet requires labels during training (semi-supervised)
semi_supervised_models = [
    {"algorithm_": "devnet", "epochs": 50, "verbose": 0},
]

deep_learning_results = []

for dataset_name, config in uci_dataset_configs.items():
    print(f"\n{'='*60}")
    print(f"Dataset: {dataset_name} - Deep Learning Models")
    print(f"Training samples: {config['train_normals']} | Test contamination: {config['benchmark_test_positive_ratio']:.3f}")
    print("="*60)

    # Standard unsupervised deep learning models
    for model_params in deep_learning_models:
        algorithm_name = model_params["algorithm_"]
        shared_kwargs = {k: v for k, v in model_params.items() if k != "algorithm_"}
        
        try:
            model_kwargs = {
                **model_params,
                "contamination": config["benchmark_test_positive_ratio"],
            }

            print(f"\n⏳ Training {algorithm_name}...", end=" ")
            
            # === RADAR Platform timing ===
            model = pyod.PyodAnomalyDetection(**model_kwargs)
            start_platform = time.time()
            model.fit(config["X_train"])
            platform_fit_time = time.time() - start_platform
            
            start_predict = time.time()
            predictions = np.asarray(model.predict(config["X_test"])).astype(int).ravel()
            scores = np.asarray(model.decision_function(config["X_test"])).ravel()
            platform_predict_time = time.time() - start_predict
            platform_total_time = platform_fit_time + platform_predict_time

            # === Direct PyOD timing ===
            direct_cls = direct_pyod_deep[algorithm_name]
            direct_kwargs = {
                "contamination": config["benchmark_test_positive_ratio"],
                **shared_kwargs
            }
            direct_model = direct_cls(**direct_kwargs)
            start_direct = time.time()
            direct_model.fit(config["X_train"])
            direct_fit_time = time.time() - start_direct
            
            start_direct_pred = time.time()
            _ = direct_model.predict(config["X_test"])
            _ = direct_model.decision_function(config["X_test"])
            direct_predict_time = time.time() - start_direct_pred
            direct_total_time = direct_fit_time + direct_predict_time

            # Calculate overhead and speedup
            overhead = platform_total_time - direct_total_time
            speedup = direct_total_time / platform_total_time if platform_total_time > 0 else np.nan

            # Metrics
            accuracy = metrics_module.metric_accuracy(config["y_test"], predictions) / 100
            precision = metrics_module.metric_precision(config["y_test"], predictions)
            recall = metrics_module.metric_recall(config["y_test"], predictions)
            f1 = metrics_module.metric_F1score(config["y_test"], predictions)

            finite_scores = np.isfinite(scores)
            if finite_scores.all():
                roc_auc = metrics_module.metric_AUC_ROC_scores(config["y_test"], scores)
                pr_auc = metrics_module.metric_PR_AUC(config["y_test"], scores)
            else:
                roc_auc = np.nan
                pr_auc = np.nan

            print(f"F1={f1:.3f}, ROC-AUC={roc_auc:.3f} | Platform: {platform_total_time:.2f}s, Direct: {direct_total_time:.2f}s, Overhead: {overhead:.2f}s")

            deep_learning_results.append({
                "dataset": dataset_name,
                "algorithm": algorithm_name,
                "category": "deep_learning",
                "contamination": round(config["benchmark_test_positive_ratio"], 4),
                "accuracy": round(accuracy, 4),
                "precision": round(precision, 4),
                "recall": round(recall, 4),
                "f1": round(f1, 4),
                "roc_auc": round(float(roc_auc), 4) if np.isfinite(roc_auc) else np.nan,
                "pr_auc": round(float(pr_auc), 4) if np.isfinite(pr_auc) else np.nan,
                "platform_time_s": round(platform_total_time, 4),
                "direct_time_s": round(direct_total_time, 4),
                "overhead_s": round(overhead, 4),
                "speedup": round(speedup, 4) if not np.isnan(speedup) else np.nan,
            })

        except Exception as e:
            print(f"Error - {str(e)[:60]}")
            deep_learning_results.append({
                "dataset": dataset_name,
                "algorithm": algorithm_name,
                "category": "deep_learning",
                "contamination": round(config["benchmark_test_positive_ratio"], 4),
                "accuracy": np.nan,
                "precision": np.nan,
                "recall": np.nan,
                "f1": np.nan,
                "roc_auc": np.nan,
                "pr_auc": np.nan,
                "platform_time_s": np.nan,
                "direct_time_s": np.nan,
                "overhead_s": np.nan,
                "speedup": np.nan,
                "error": str(e)[:100],
            })

deep_learning_df = pd.DataFrame(deep_learning_results).sort_values(
    ["dataset", "f1"], ascending=[True, False]
).reset_index(drop=True)

print("\n" + "="*60)
print("Results Summary - Deep Learning Models")
print("="*60)
display(deep_learning_df)

### Combined Results: All Models Comparison

In [ ]:
# Combine all results: original + new classical + deep learning
# All dataframes now have the same structure with timing and overhead columns

# Combine all dataframes
all_results_df = pd.concat([
    uci_results_df,
    new_classical_df,
    deep_learning_df
], ignore_index=True)

# Sort by dataset and F1 score
all_results_df = all_results_df.sort_values(
    ["dataset", "f1"], ascending=[True, False]
).reset_index(drop=True)

print("Top 10 Models by F1 Score (per dataset)")
print("="*60)
for dataset in all_results_df["dataset"].unique():
    print(f"\n{dataset.upper()}")
    top_models = all_results_df[all_results_df["dataset"] == dataset].head(10)
    display(top_models[["algorithm", "category", "f1", "roc_auc", "pr_auc", "platform_time_s", "overhead_s", "speedup"]])

# Overhead analysis
print("\n" + "="*60)
print("Timing Analysis: RADAR Platform vs Direct PyOD")
print("="*60)
print("\n• speedup > 1: Direct PyOD is faster than RADAR platform")
print("• speedup < 1: RADAR platform is faster than Direct PyOD")
print("• speedup ≈ 1: Both have similar performance")
print("• overhead > 0: RADAR adds overhead (slower)")
print("• overhead < 0: RADAR is faster (negative overhead)\n")

for dataset in all_results_df["dataset"].unique():
    print(f"\n{dataset.upper()} - Overhead Analysis")
    dataset_df = all_results_df[all_results_df["dataset"] == dataset].copy()
    # Sort by overhead (lowest first = best RADAR performance)
    dataset_df = dataset_df.sort_values("overhead_s", ascending=True)
    display(dataset_df[["algorithm", "category", "platform_time_s", "direct_time_s", "overhead_s", "speedup", "f1"]])

# Overall ranking
print("\n" + "="*60)
print("Complete Results Table (with timing & overhead)")
print("="*60)
display(all_results_df)

In [ ]:
# Save all results (including timing and overhead comparison)
# Uses results_dir defined in the first cell (project_root / 'results')

# Save individual results with timing
original_path = results_dir / "uci_pyod_original_results.csv"
new_classical_path = results_dir / "uci_pyod_new_classical_results.csv"
deep_learning_path = results_dir / "uci_pyod_deep_learning_results.csv"
all_results_path = results_dir / "uci_pyod_all_models_results.csv"

uci_results_df.to_csv(original_path, index=False)
new_classical_df.to_csv(new_classical_path, index=False)
deep_learning_df.to_csv(deep_learning_path, index=False)
all_results_df.to_csv(all_results_path, index=False)

print(f"Saved original models results to: {original_path}")
print(f"Saved new classical results to: {new_classical_path}")
print(f"Saved deep learning results to: {deep_learning_path}")
print(f"Saved combined results to: {all_results_path}")
print(f"\nAll CSVs include:")
print("   - Metrics: accuracy, precision, recall, f1, roc_auc, pr_auc")
print("   - Timing: platform_time_s, direct_time_s, overhead_s, speedup")